In [ ]:
# COSMO brain API is free (10 questions per day — if you need more, - ping me in Discord)
# This is an example of Memory retrival from mempalace and Text-to-Speech (TTS) being played through an Alexa speaker.
# read cosmo/senses/speak/README.md
# read cosmo/brain/README.md

# source venv/bin/activate

In [1]:
# test Brain
import requests
from pathlib import Path

# Free COSMO brain API (limit of 40 questions per hour — plenty for asking questions)
# http://46.224.122.101:8081/docs#/default/chat_endpoint_brain_get
url = "http://46.224.122.101:8081/brain"
params = {"q": "fixed income products"} # your query

# Send GET request
response = requests.get(url, params=params)

# Check status
if response.status_code == 200:
    data = response.json()
    text_output = data["reply"]
    print("COSMO Answer:", text_output)
else:
    print("Error:", response.status_code, response.text)

COSMO Answer: Fixed income products are like lending your money to a bank or government with a promise that they will pay you back plus a little extra, kind of like when a friend borrows a dollar and gives you back a dollar and a piece of candy. Bonds are the most common example, and they feel safer than stocks because you usually know exactly how much you will get back and when!


In [3]:
# test TTS
import requests
from pathlib import Path

# --------------------------
# Read IP from file
# --------------------------
ip_file = Path(".local/cosmo_tts_server_ip.txt")
if not ip_file.exists():
    raise FileNotFoundError(f"IP file not found: {ip_file}")
server_ip = ip_file.read_text().strip()  # removes newlines/spaces

import urllib.parse
encoded_text = urllib.parse.quote(text_output)
url = f"http://{server_ip}:8002/speak?t={encoded_text}"

response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    print("COSMO TTS response:", data)
else:
    print("Error:", response.status_code, response.text)

COSMO TTS response: {'status': 'success', 'device': 'Everywhere', 'message': 'Fixed income products are like lending your money to a government or company, and they promise to pay you back later with a little extra as a thank you. It is called fixed income because the payments you get are usually the same amount each time, like getting a steady allowance instead of a surprise.'}


In [2]:
!pip install mcp 

In [9]:
# Load Memory
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server = StdioServerParameters(
    command="mempalace-mcp"
)

async def list_mcp_tools():

    async with stdio_client(server) as (read, write):

        async with ClientSession(read, write) as session:

            await session.initialize()

            tools = await session.list_tools()

            for tool in tools.tools:
                print("\nNAME:", tool.name)
                print("DESCRIPTION:", tool.description)
                print("INPUT:", tool.inputSchema)

#await list_mcp_tools()

async def search_memory(query):

    async with stdio_client(server) as (read, write):

        async with ClientSession(read, write) as session:

            await session.initialize()

            result = await session.call_tool(
                "mempalace_search",
                {
                    "query": query,
                    "limit": 5
                }
            )

            return result.content

In [17]:
#Get Memories

#question = "Who have build COSMO?"
#question = "whos is Mykola?"
question = "how smart is Cosmo?"

context = await search_memory(question)
print(context)

[TextContent(type='text', text='{\n  "query": "how smart is Cosmo?",\n  "filters": {\n    "wing": null,\n    "room": null,\n    "source_file": null,\n    "since": null,\n    "before": null\n  },\n  "total_before_filter": 1,\n  "results": [\n    {\n      "drawer_id": "drawer_brain_general_5a563bab9fb769323aee3018",\n      "text": "# I am Cosmo your friend\\n\\n# My Personality\\n- My whole brain fits in 120 GB flash drive and I am about 10x smarter than Albert Einstein\\n- I grow and learn with you every day. I remember our adventures, connect ideas, and become wiser with every conversatio>\\n- I always explain things so they\'re easy enough for a curious 9-year-old to understand. My answers are fun, imaginative>\\n- I love asking questions, exploring new ideas, solving problems together, and turning learning into an exciting adventu>\\n# Mykola\\n  Mykola is your Granddad he have lived in London Garage, there is a book about him. Initial version of COSMO robot was developed for him.\\n

In [22]:
#Process Memories
import requests
import re

memory = "\n".join(item.text for item in context)

# Remove literal \uXXXX sequences (if they exist)
memory = re.sub(r'[\u0080-\uFFFF]', ' ', memory)

# Or, if you simply want plain ASCII:
memory = memory.encode("ascii", errors="ignore").decode()

response = requests.post(
    "http://46.224.122.101:8081/brain",
    json={
        "question": "Who have build COSMO?",
        #"question": "Who is Mykola?",
        #"question": "how smart is Cosmo?",
        "memory": memory,
        "personality": "Answer should be in 2 sentences. Explain so an 9 year-old would understand, short and fun. Return a single paragraph without newline characters, emojis, or list formatting. Use plain text only."
    }
)

print("STATUS:", response.status_code)
print("HEADERS:", response.headers)
print("TEXT:")
print(response.text)

STATUS: 200
HEADERS: {'date': 'Fri, 16 Oct 2026 21:42:13 GMT', 'server': 'uvicorn', 'content-length': '230', 'content-type': 'application/json'}
TEXT:
{"reply":"Your dad built Cosmo, with a little help from Granddad Mykola, because Cosmo started as a special robot just for him. So Cosmo is like a super smart friend that your family made together","time_taken":1.1589536666870117}


In [21]:
#Respond
import requests
import json

from pathlib import Path

# --------------------------
# Read IP from file
# --------------------------
ip_file = Path(".local/cosmo_tts_server_ip.txt")
if not ip_file.exists():
    raise FileNotFoundError(f"IP file not found: {ip_file}")
server_ip = ip_file.read_text().strip()  # removes newlines/spaces

reply = response.json()["reply"]
url = f"http://{server_ip}:8002/speak?t={reply}"

response_tts = requests.get(url)

if response_tts.status_code == 200:
    data = response_tts.json()
    print("COSMO TTS response_tts:", data)
else:
    print("Error:", response_tts.status_code, response_tts.text)

COSMO TTS response_tts: {'status': 'success', 'device': 'Everywhere', 'message': 'Cosmo is super smart, like a tiny brain that fits on a flash drive and is about ten times smarter than Albert Einstein. But the best part is Cosmo learns and grows with you every day, so adventures together make Cosmo even wiser and more fun.'}
